In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
from sklearn.preprocessing import LabelEncoder
import numpy as np
from tqdm import tqdm
import os
from torch.cuda.amp import GradScaler
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, accuracy_score, classification_report
import datetime
import json
import random
import matplotlib.pyplot as plt
import seaborn as sns
import csv
import time
import sys

#  配置 
CONFIG = {
    'seed': 42,
    'train_csv': "data/train.csv",
    'val_csv': "data/val.csv",
    'test_csv': "data/test.csv",
    'batch_size': 128,
    'num_epochs': 100,
    'learning_rate': 0.001,
    'weight_decay': 0.02,
    'early_stop_patience': 20,
    'max_seq_len_percentile': 90,  
    'embed_dim': 128,
    'hidden_dim': 320,
    'num_lstm_layers': 2,
    'use_warmup': True,
    'warmup_epochs': 10,
    'cosine_T_max': 90,
    'cosine_eta_min': 1e-5,
    'max_grad_norm': 1.0,
}

def set_seed(seed):
    """设置随机种子"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# 数据增强模块
class DataAugmentation:
    """数据增强策略 - 直接应用在数据集中"""
    
    @staticmethod
    def apply(sequence, augmentation_strength=0.2):
        """应用组合增强"""
        augmented = sequence
        
        # 随机决定是否应用每种增强
        if random.random() < augmentation_strength:
            augmented = DataAugmentation.random_mutation(augmented, rate=0.02)
        
        if random.random() < augmentation_strength:
            augmented = DataAugmentation.random_deletion(augmented, rate=0.01)
        
        if random.random() < augmentation_strength:
            augmented = DataAugmentation.random_insertion(augmented, rate=0.01)
        
        if random.random() < augmentation_strength:
            augmented = DataAugmentation.random_swap(augmented, rate=0.02)
        
        if random.random() < augmentation_strength * 0.5:
            augmented = DataAugmentation.reverse_complement(augmented)
        
        return augmented
    
    @staticmethod
    def random_insertion(seq, rate=0.01):
        """随机插入碱基"""
        bases = ['A', 'C', 'G', 'T']
        if len(seq) <= 10:
            return seq
        seq_list = list(seq)
        insert_count = max(1, int(len(seq) * rate))
        
        for _ in range(insert_count):
            pos = random.randint(0, len(seq_list))
            seq_list.insert(pos, random.choice(bases))
        
        return ''.join(seq_list)
    
    @staticmethod
    def reverse_complement(seq):
        """反向互补"""
        complement = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C', 'N': 'N'}
        return ''.join([complement.get(c, c) for c in seq[::-1]])
    
    @staticmethod
    def random_mutation(seq, rate=0.02):
        """随机突变"""
        bases = ['A', 'C', 'G', 'T', 'N']
        seq_list = list(seq)
        for i in range(len(seq_list)):
            if random.random() < rate:
                seq_list[i] = random.choice(bases)
        return ''.join(seq_list)
    
    @staticmethod
    def random_deletion(seq, rate=0.01):
        """随机删除"""
        if len(seq) <= 10:
            return seq
        seq_list = list(seq)
        delete_count = max(1, int(len(seq) * rate))
        for _ in range(delete_count):
            if len(seq_list) > 5:
                pos = random.randint(0, len(seq_list) - 1)
                seq_list.pop(pos)
        return ''.join(seq_list)
    
    @staticmethod
    def random_swap(seq, rate=0.02):
        """随机交换相邻碱基"""
        if len(seq) <= 2:
            return seq
        seq_list = list(seq)
        swap_count = max(1, int(len(seq) * rate))
        for _ in range(swap_count):
            if len(seq_list) > 1:
                i = random.randint(0, len(seq_list) - 2)
                seq_list[i], seq_list[i + 1] = seq_list[i + 1], seq_list[i]
        return ''.join(seq_list)

#残差连接模块
class ResidualBlock(nn.Module):
    """残差块"""
    def __init__(self, channels, kernel_size=3):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=kernel_size//2)
        self.bn1 = nn.BatchNorm1d(channels)
        self.activation1 = nn.GELU()
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=kernel_size//2)
        self.bn2 = nn.BatchNorm1d(channels)
        self.activation2 = nn.GELU()
    
    def forward(self, x):
        residual = x
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.activation1(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = x + residual
        x = self.activation2(x)
        return x

# 1D LKA模块 (大核注意力)
class LKA1D(nn.Module):
    """大核注意力模块"""
    def __init__(self, dim):
        super().__init__()
        # 5x1卷积，用于局部特征提取
        self.conv0 = nn.Conv1d(dim, dim, kernel_size=5, padding=2, groups=dim)
        # 7x1空洞卷积，用于扩大感受野
        self.conv_spatial = nn.Conv1d(dim, dim, kernel_size=7, stride=1, padding=9, 
                                     groups=dim, dilation=3)
        # 1x1卷积，用于特征融合和通道调整
        self.conv1 = nn.Conv1d(dim, dim, kernel_size=1)
        
    def forward(self, x):
        # 保留原始输入
        u = x.clone()
        # 大核注意力计算
        attn = self.conv0(x)
        attn = self.conv_spatial(attn)
        attn = self.conv1(attn)
        # 残差连接 + 注意力加权
        return u * attn

# 嵌入层 
class DNAEmbedding(nn.Module):
    """DNA序列嵌入层"""
    def __init__(self, vocab_size=5, embed_dim=128, max_len=2000):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.max_len = max_len
        
        # 1. 基础投影层
        self.base_projection = nn.Sequential(
            nn.Linear(vocab_size, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.GELU(),
            nn.Dropout(0.1)
        )
        
        # 2. 位置编码（可学习）
        self.position_encoding = nn.Parameter(torch.randn(1, max_len, embed_dim) * 0.02)
        
        # 3. 化学性质感知卷积（捕捉局部模式）
        self.chem_conv = nn.Sequential(
            nn.Conv1d(embed_dim, embed_dim, kernel_size=5, padding=2, groups=4),
            nn.BatchNorm1d(embed_dim),
            nn.GELU(),
            nn.Dropout(0.05),
            nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(embed_dim),
            nn.GELU(),
            nn.Dropout(0.05)
        )
        
        # 4. 层归一化
        self.layer_norm = nn.LayerNorm(embed_dim)
        
        # 5. 最终dropout
        self.final_dropout = nn.Dropout(0.1)
        
    def forward(self, x_onehot):
        """
        x_onehot: [batch_size, seq_len, vocab_size]
        返回: [batch_size, seq_len, embed_dim]
        """
        batch_size, seq_len, _ = x_onehot.shape
        
        # 重塑以适用于BatchNorm1d
        x_flat = x_onehot.reshape(-1, self.vocab_size)
        base_emb = self.base_projection(x_flat)
        base_emb = base_emb.reshape(batch_size, seq_len, self.embed_dim)
        
        # 添加位置编码
        pos_emb = self.position_encoding[:, :seq_len, :]
        emb = base_emb + pos_emb
        
        # 化学性质感知卷积
        emb_transposed = emb.transpose(1, 2)
        chem_enhanced = self.chem_conv(emb_transposed)
        chem_enhanced = chem_enhanced.transpose(1, 2)
        
        # 残差连接 + 层归一化
        emb = emb + chem_enhanced
        emb = self.layer_norm(emb)
        
        return self.final_dropout(emb)

#  DeepPlantTE多尺度CNN模块
class MultiScaleCNN(nn.Module):
    """DeepPlantTE多尺度CNN模块"""
    def __init__(self, in_channels, out_channels_per_kernel=64):
        super(MultiScaleCNN, self).__init__()
        
        # 瓶颈层
        self.bottleneck = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=1, padding=0),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.1)
        )
        
        # 多尺度卷积核
        kernel_sizes = [3, 5, 7, 9, 11, 13, 15]
        
        # 输出通道数
        self.out_channels = len(kernel_sizes) * out_channels_per_kernel
        
        # 创建多尺度卷积层
        self.convs = nn.ModuleList()
        for kernel_size in kernel_sizes:
            conv = nn.Sequential(
                nn.Conv1d(64, out_channels_per_kernel, kernel_size=kernel_size, 
                         padding=kernel_size // 2),
                nn.BatchNorm1d(out_channels_per_kernel),
                nn.GELU(),
                ResidualBlock(out_channels_per_kernel, kernel_size=3),
                nn.Dropout(0.15)
            )
            self.convs.append(conv)
    
    def forward(self, x):
        # 通过瓶颈层
        x = self.bottleneck(x)
        
        conv_outputs = []
        for conv in self.convs:
            out = conv(x)
            conv_outputs.append(out)
        
        out = torch.cat(conv_outputs, dim=1)
        return out

#  DeepPlantTE模型架构
class DeepPlantTE(nn.Module):
    """DeepPlantTE模型 """
    def __init__(self, vocab_size=5, embed_dim=128, hidden_dim=320, num_classes=10, 
                 num_lstm_layers=2, max_seq_len=2000):
        super(DeepPlantTE, self).__init__()
        
        self.vocab_size = vocab_size
        self.num_classes = num_classes
        self.hidden_dim = hidden_dim
        
        #嵌入层
        self.embedding = DNAEmbedding(
            vocab_size=vocab_size,
            embed_dim=embed_dim,
            max_len=max_seq_len
        )
        
        # CNN-BiLSTM分支 
        self.multi_scale_cnn = MultiScaleCNN(embed_dim, out_channels_per_kernel=64)
        cnn_output_dim = self.multi_scale_cnn.out_channels  
        
        # LKA-1D注意力模块 
        self.lka = LKA1D(cnn_output_dim)
        
        self.cnn_dropout = nn.Dropout(0.15)
        
        self.bilstm = nn.LSTM(
            cnn_output_dim, hidden_dim // 2, num_layers=num_lstm_layers,
            bidirectional=True, batch_first=True, 
            dropout=0.15 if num_lstm_layers > 1 else 0
        )
        self.bilstm_dropout = nn.Dropout(0.15)
        
        self.attention_gate = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
        
        # 特征融合
        fusion_input_dim = hidden_dim
        
        self.fusion_projection = nn.Sequential(
            nn.Linear(fusion_input_dim, hidden_dim * 2), 
            nn.BatchNorm1d(hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        
        # 特征头
        self.feature_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim * 2),  
            nn.BatchNorm1d(hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(hidden_dim * 2, hidden_dim),  
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.15),
        )
        
        #分类头 
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), 
            nn.BatchNorm1d(hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(hidden_dim // 2, hidden_dim // 4),  
            nn.BatchNorm1d(hidden_dim // 4),
            nn.GELU(),
            nn.Dropout(0.08),
            nn.Linear(hidden_dim // 4, num_classes)  
        )
    
    def apply_attention_mask(self, attn_weights, seq_lengths, max_len):
        batch_size = attn_weights.shape[0]
        mask = torch.arange(max_len, device=attn_weights.device).unsqueeze(0) < seq_lengths.unsqueeze(1)
        
        if attn_weights.dtype == torch.float16:
            mask_value = torch.tensor(-65500.0, dtype=torch.float16, device=attn_weights.device)
        else:
            mask_value = torch.tensor(-1e9, dtype=torch.ones_like(attn_weights).dtype, device=attn_weights.device)
        
        attn_weights = attn_weights.masked_fill(~mask, mask_value)
        return attn_weights
    
    def forward(self, x_full, full_seq_len=None):
        batch_size, seq_len, vocab_size = x_full.shape
        
        # 嵌入处理
        embedded = self.embedding(x_full)
        
        # 转置以适配CNN
        embedded_t = embedded.transpose(1, 2)
        
        # CNN多尺度特征提取
        cnn_features = self.multi_scale_cnn(embedded_t)
        
        # 应用LKA-1D大核注意力
        cnn_features = self.lka(cnn_features)
        
        cnn_features = self.cnn_dropout(cnn_features)
        cnn_features_t = cnn_features.transpose(1, 2)
        
        # CNN-BiLSTM时序建模
        bilstm_out, _ = self.bilstm(cnn_features_t)
        attn_weights = self.attention_gate(bilstm_out)
        attn_weights = attn_weights.squeeze(-1)
        
        if full_seq_len is not None:
            attn_weights = self.apply_attention_mask(
                attn_weights, full_seq_len, bilstm_out.shape[1]
            )
        
        attn_weights = torch.softmax(attn_weights, dim=1)
        context = torch.sum(bilstm_out * attn_weights.unsqueeze(-1), dim=1)
        context = self.bilstm_dropout(context)
        
        # 特征融合
        fused_features = self.fusion_projection(context)
        
        # 分类
        features = self.feature_head(fused_features)
        logits = self.classifier(features)
        
        return logits, features

# 数据集
class DeepPlantTEDataset(Dataset):
    """DeepPlantTE数据集类"""
    def __init__(self, csv_file, max_seq_len, label_encoder=None, augment=False):
        df = pd.read_csv(csv_file)
        self.sequences = df['Sequence'].values
        self.labels = df['Mapped_Category2'].values
        self.augment = augment
        self.max_seq_len = max_seq_len
        
        if label_encoder is None:
            self.label_encoder = LabelEncoder()
            self.labels = self.label_encoder.fit_transform(self.labels)
        else:
            self.label_encoder = label_encoder
            self.labels = self.label_encoder.transform(self.labels)
        
        self.char_to_idx = {'A': 0, 'C': 1, 'G': 2, 'T': 3, 'N': 4}
        self.vocab_size = len(self.char_to_idx)
    
    def __len__(self):
        return len(self.sequences)
    
    def sequence_to_onehot(self, seq, max_len):
        """将序列转换为one-hot编码"""
        seq_encoded = [self.char_to_idx.get(c, self.char_to_idx['N']) for c in seq]
        
        # 截断或填充
        if len(seq_encoded) > max_len:
            seq_encoded = seq_encoded[:max_len]
        else:
            seq_encoded += [self.char_to_idx['N']] * (max_len - len(seq_encoded))
        
        onehot = np.zeros((max_len, self.vocab_size), dtype=np.float32)
        for i, idx in enumerate(seq_encoded):
            onehot[i, idx] = 1.0
        
        return onehot
    
    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]
        
        # 应用数据增强
        if self.augment:
            seq = DataAugmentation.apply(seq, augmentation_strength=0.2)
        
        full_encoded = self.sequence_to_onehot(seq, self.max_seq_len)
        
        return (torch.tensor(full_encoded, dtype=torch.float32),
                torch.tensor(label, dtype=torch.long),
                torch.tensor(len(seq), dtype=torch.long))

# 训练过程监控
class TrainingMonitor:
    """训练监控器"""
    def __init__(self):
        self.grad_norms = []
        self.learning_rates = []
    
    def track_gradients(self, model):
        total_norm = 0
        for p in model.parameters():
            if p.grad is not None:
                param_norm = p.grad.data.norm(2)
                total_norm += param_norm.item() ** 2
        total_norm = total_norm ** 0.5
        self.grad_norms.append(total_norm)
        
    def track_learning_rate(self, optimizer):
        current_lr = optimizer.param_groups[0]['lr']
        self.learning_rates.append(current_lr)

#辅助函数
def compute_metrics(true_labels, pred_labels, num_classes):
    """计算评估指标"""
    true_labels = np.array(true_labels)
    pred_labels = np.array(pred_labels)
    
    acc = accuracy_score(true_labels, pred_labels)
    
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        true_labels, pred_labels, average='macro', zero_division=0
    )
    
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        true_labels, pred_labels, average='weighted', zero_division=0
    )
    
    precision_per_class, recall_per_class, f1_per_class, support_per_class = precision_recall_fscore_support(
        true_labels, pred_labels, average=None, zero_division=0
    )
    
    cm = confusion_matrix(true_labels, pred_labels, labels=range(num_classes))
    
    class_report = classification_report(true_labels, pred_labels, 
                                         target_names=[f'Class_{i}' for i in range(num_classes)],
                                         output_dict=True, zero_division=0)
    
    return {
        'accuracy': acc,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro,
        'precision_weighted': precision_weighted,
        'recall_weighted': recall_weighted,
        'f1_weighted': f1_weighted,
        'precision_per_class': precision_per_class.tolist(),
        'recall_per_class': recall_per_class.tolist(),
        'f1_per_class': f1_per_class.tolist(),
        'support_per_class': support_per_class.tolist(),
        'confusion_matrix': cm,
        'classification_report': class_report
    }, cm

#  训练函数
def train_epoch(model, train_loader, criterion, optimizer, device, scaler, epoch,
                num_classes, label_encoder, total_epochs, results_dir=None, monitor=None):
    """训练一个epoch"""
    model.train()
    total_raw_loss = 0
    all_preds = []
    all_labels = []
    correct = 0
    total = 0
    skipped_batches = 0
    skipped_grad_steps = 0
    
    progress_bar = tqdm(train_loader, desc="Training")
    
    for batch_idx, batch in enumerate(progress_bar):
        sequences_full, labels, full_seq_len = batch
        full_seq_len = full_seq_len.to(device)
        
        sequences_full = sequences_full.to(device)
        labels = labels.to(device)
        
        try:
            if device.type == 'cuda':
                with torch.amp.autocast('cuda'):
                    logits, features = model(sequences_full, full_seq_len)
                    cls_loss = criterion(logits, labels)
                    total_loss_batch = cls_loss
                    loss_for_backward = total_loss_batch
            else:
                logits, features = model(sequences_full, full_seq_len)
                cls_loss = criterion(logits, labels)
                total_loss_batch = cls_loss
                loss_for_backward = total_loss_batch
            
            if torch.isnan(total_loss_batch):
                skipped_batches += 1
                optimizer.zero_grad()
                continue
            
            total_raw_loss += total_loss_batch.item()
            
            scaler.scale(loss_for_backward).backward()
            
            try:
                scaler.unscale_(optimizer)
                grad_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), 
                    CONFIG['max_grad_norm'],
                    error_if_nonfinite=False
                )
                
                if monitor:
                    monitor.track_gradients(model)
                
                if torch.isnan(grad_norm) or torch.isinf(grad_norm) or grad_norm > 100:
                    print(f"\n⚠ Skipped step due to abnormal grad: {grad_norm}")
                    skipped_grad_steps += 1
                    optimizer.zero_grad()
                else:
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
                    
            except RuntimeError as e:
                error_str = str(e)
                if "unscale_() has already been called" in error_str:
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
                elif "Found inf" in error_str or "Found NaN" in error_str:
                    print(f"\n⚠ Skipped step due to inf/NaN in gradients")
                    skipped_grad_steps += 1
                    optimizer.zero_grad()
                    scaler.update()
                else:
                    print(f"\n⚠ Unexpected error in optimization: {error_str}")
                    optimizer.zero_grad()
                    scaler.update()
            
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"\n⚠ OOM detected, skipping batch {batch_idx}")
                torch.cuda.empty_cache()
                optimizer.zero_grad()
                skipped_batches += 1
                continue
            else:
                print(f"\n⚠ RuntimeError: {e}")
                optimizer.zero_grad()
                skipped_batches += 1
                continue
        
        _, predicted = torch.max(logits.detach(), 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        current_lr = optimizer.param_groups[0]['lr']
        avg_loss_so_far = total_raw_loss / (batch_idx + 1)
        
        batch_correct = (predicted == labels).sum().item()
        batch_acc = batch_correct / labels.size(0) * 100
        
        progress_bar.set_postfix({
            'loss': f'{avg_loss_so_far:.4f}',
            'acc': f'{correct / total * 100:.2f}%',
            'batch_acc': f'{batch_acc:.1f}%',
            'lr': f'{current_lr:.6f}',
            'skip': f'G:{skipped_grad_steps}/B:{skipped_batches}'
        })
    
    avg_loss = total_raw_loss / len(train_loader) if len(train_loader) > 0 else 0
    
    metrics, cm = compute_metrics(all_labels, all_preds, num_classes)
    
    skip_info = ''
    if skipped_batches > 0 or skipped_grad_steps > 0:
        skip_info = f' (skipped {skipped_batches} batches, {skipped_grad_steps} grad steps)'
    
    print(f"\n训练详细指标 (Epoch {epoch + 1}):")
    print(f"  {'='*50}")
    print(f"  损失: {avg_loss:.6f}")
    print(f"  准确率: {metrics['accuracy']*100:.4f}%")
    print(f"  {'-'*30}")
    print(f"  宏平均指标:")
    print(f"    精确率: {metrics['precision_macro']:.6f}")
    print(f"    召回率: {metrics['recall_macro']:.6f}")
    print(f"    F1分数: {metrics['f1_macro']:.6f}")
    print(f"  {'-'*30}")
    print(f"  加权平均指标:")
    print(f"    精确率: {metrics['precision_weighted']:.6f}")
    print(f"    召回率: {metrics['recall_weighted']:.6f}")
    print(f"    F1分数: {metrics['f1_weighted']:.6f}")
    
    if num_classes <= 20:
        print(f"  {'-'*30}")
        print(f"  每个类别指标:")
        for i in range(num_classes):
            class_name = label_encoder.classes_[i] if i < len(label_encoder.classes_) else f'Class_{i}'
            print(f"    {class_name:20s}: 精确率={metrics['precision_per_class'][i]:.6f}, "
                  f"召回率={metrics['recall_per_class'][i]:.6f}, "
                  f"F1={metrics['f1_per_class'][i]:.6f}, "
                  f"样本数={metrics['support_per_class'][i]}")
    
    log_info = (f'Epoch {epoch + 1:3d} Train - Loss: {avg_loss:.6f}, '
                f'Acc: {metrics["accuracy"]:.6f}, '
                f'F1_macro: {metrics["f1_macro"]:.6f}, '
                f'F1_weighted: {metrics["f1_weighted"]:.6f}, '
                f'Precision_macro: {metrics["precision_macro"]:.6f}, '
                f'Recall_macro: {metrics["recall_macro"]:.6f}{skip_info}\n')
    
    if results_dir:
        with open(os.path.join(results_dir, 'train_log.txt'), 'a') as f:
            f.write(log_info)
        
        epoch_metrics_file = os.path.join(results_dir, 'train_epoch_metrics.csv')
        epoch_metrics = {
            'epoch': epoch + 1,
            'loss': avg_loss,
            'accuracy': metrics['accuracy'],
            'precision_macro': metrics['precision_macro'],
            'recall_macro': metrics['recall_macro'],
            'f1_macro': metrics['f1_macro'],
            'precision_weighted': metrics['precision_weighted'],
            'recall_weighted': metrics['recall_weighted'],
            'f1_weighted': metrics['f1_weighted']
        }
        
        for i in range(num_classes):
            class_name = label_encoder.classes_[i] if i < len(label_encoder.classes_) else f'Class_{i}'
            epoch_metrics[f'f1_{class_name}'] = metrics['f1_per_class'][i]
            epoch_metrics[f'precision_{class_name}'] = metrics['precision_per_class'][i]
            epoch_metrics[f'recall_{class_name}'] = metrics['recall_per_class'][i]
        
        file_exists = os.path.isfile(epoch_metrics_file)
        with open(epoch_metrics_file, 'a', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=epoch_metrics.keys())
            if not file_exists:
                writer.writeheader()
            writer.writerow(epoch_metrics)
    
    return avg_loss, metrics

# 验证函数
def validate(model, val_loader, criterion, device, epoch, num_classes, label_encoder, total_epochs, results_dir=None):
    """验证函数"""
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in val_loader:
            sequences_full, labels, full_seq_len = batch
            full_seq_len = full_seq_len.to(device)
            
            sequences_full = sequences_full.to(device)
            labels = labels.to(device)
            
            if device.type == 'cuda':
                with torch.amp.autocast('cuda'):
                    logits, features = model(sequences_full, full_seq_len)
                    cls_loss = criterion(logits, labels)
                    total_weighted_loss = cls_loss
            else:
                logits, features = model(sequences_full, full_seq_len)
                cls_loss = criterion(logits, labels)
                total_weighted_loss = cls_loss
            
            total_loss += total_weighted_loss.item()
            
            _, predicted = torch.max(logits, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(val_loader)
    
    metrics, cm = compute_metrics(all_labels, all_preds, num_classes)
    
    print(f"\n验证详细指标 (Epoch {epoch + 1}):")
    print(f"  {'='*50}")
    print(f"  损失: {avg_loss:.6f}")
    print(f"  准确率: {metrics['accuracy']*100:.4f}%")
    print(f"  {'-'*30}")
    print(f"  宏平均指标:")
    print(f"    精确率: {metrics['precision_macro']:.6f}")
    print(f"    召回率: {metrics['recall_macro']:.6f}")
    print(f"    F1分数: {metrics['f1_macro']:.6f}")
    print(f"  {'-'*30}")
    print(f"  加权平均指标:")
    print(f"    精确率: {metrics['precision_weighted']:.6f}")
    print(f"    召回率: {metrics['recall_weighted']:.6f}")
    print(f"    F1分数: {metrics['f1_weighted']:.6f}")
    
    if num_classes <= 20:
        print(f"  {'-'*30}")
        print(f"  每个类别指标:")
        for i in range(num_classes):
            class_name = label_encoder.classes_[i] if i < len(label_encoder.classes_) else f'Class_{i}'
            print(f"    {class_name:20s}: 精确率={metrics['precision_per_class'][i]:.6f}, "
                  f"召回率={metrics['recall_per_class'][i]:.6f}, "
                  f"F1={metrics['f1_per_class'][i]:.6f}, "
                  f"样本数={metrics['support_per_class'][i]}")
    
    log_info = (f'Epoch {epoch + 1:3d} Val - Loss: {avg_loss:.6f}, '
                f'Acc: {metrics["accuracy"]:.6f}, '
                f'F1_macro: {metrics["f1_macro"]:.6f}, '
                f'F1_weighted: {metrics["f1_weighted"]:.6f}, '
                f'Precision_macro: {metrics["precision_macro"]:.6f}, '
                f'Recall_macro: {metrics["recall_macro"]:.6f}\n')
    
    if results_dir:
        with open(os.path.join(results_dir, 'val_log.txt'), 'a') as f:
            f.write(log_info)
        
        epoch_metrics_file = os.path.join(results_dir, 'val_epoch_metrics.csv')
        epoch_metrics = {
            'epoch': epoch + 1,
            'loss': avg_loss,
            'accuracy': metrics['accuracy'],
            'precision_macro': metrics['precision_macro'],
            'recall_macro': metrics['recall_macro'],
            'f1_macro': metrics['f1_macro'],
            'precision_weighted': metrics['precision_weighted'],
            'recall_weighted': metrics['recall_weighted'],
            'f1_weighted': metrics['f1_weighted']
        }
        
        for i in range(num_classes):
            class_name = label_encoder.classes_[i] if i < len(label_encoder.classes_) else f'Class_{i}'
            epoch_metrics[f'f1_{class_name}'] = metrics['f1_per_class'][i]
            epoch_metrics[f'precision_{class_name}'] = metrics['precision_per_class'][i]
            epoch_metrics[f'recall_{class_name}'] = metrics['recall_per_class'][i]
        
        file_exists = os.path.isfile(epoch_metrics_file)
        with open(epoch_metrics_file, 'a', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=epoch_metrics.keys())
            if not file_exists:
                writer.writeheader()
            writer.writerow(epoch_metrics)
    
    return avg_loss, metrics

#  测试函数
def test(model, test_loader, criterion, device, num_classes, label_encoder, total_epochs, results_dir=None):
    """测试函数"""
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for batch in test_loader:
            sequences_full, labels, full_seq_len = batch
            full_seq_len = full_seq_len.to(device)
            
            sequences_full = sequences_full.to(device)
            labels = labels.to(device)
            
            if device.type == 'cuda':
                with torch.amp.autocast('cuda'):
                    logits, features = model(sequences_full, full_seq_len)
                    cls_loss = criterion(logits, labels)
                    total_weighted_loss = cls_loss
            else:
                logits, features = model(sequences_full, full_seq_len)
                cls_loss = criterion(logits, labels)
                total_weighted_loss = cls_loss
            
            total_loss += total_weighted_loss.item()
            
            probs = F.softmax(logits, dim=1)
            _, predicted = torch.max(logits, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    avg_loss = total_loss / len(test_loader)
    metrics, cm = compute_metrics(all_labels, all_preds, num_classes)
    
    print(f"\n{'='*80}")
    print("测试结果:")
    print(f"{'='*80}")
    print(f"损失: {avg_loss:.6f}")
    print(f"准确率: {metrics['accuracy']*100:.4f}%")
    
    print(f"\n综合指标:")
    print(f"  宏平均指标:")
    print(f"    精确率: {metrics['precision_macro']:.6f}")
    print(f"    召回率: {metrics['recall_macro']:.6f}")
    print(f"    F1分数: {metrics['f1_macro']:.6f}")
    print(f"  加权平均指标:")
    print(f"    精确率: {metrics['precision_weighted']:.6f}")
    print(f"    召回率: {metrics['recall_weighted']:.6f}")
    print(f"    F1分数: {metrics['f1_weighted']:.6f}")
    
    print(f"\n每个类别的详细指标:")
    for i in range(num_classes):
        class_name = label_encoder.classes_[i] if i < len(label_encoder.classes_) else f'Class {i}'
        print(f"  {class_name:20s}: 精确率={metrics['precision_per_class'][i]:.6f}, "
              f"召回率={metrics['recall_per_class'][i]:.6f}, "
              f"F1={metrics['f1_per_class'][i]:.6f}, "
              f"样本数={metrics['support_per_class'][i]}")
    
    log_info = (f"测试结果:\n"
                f"损失: {avg_loss:.6f}\n"
                f"准确率: {metrics['accuracy']*100:.4f}%\n"
                f"F1_macro: {metrics['f1_macro']:.6f}\n"
                f"F1_weighted: {metrics['f1_weighted']:.6f}\n"
                f"精确率_macro: {metrics['precision_macro']:.6f}\n"
                f"召回率_macro: {metrics['recall_macro']:.6f}\n\n")
    
    if results_dir:
        with open(os.path.join(results_dir, 'test_results.txt'), 'w') as f:
            f.write(log_info)
            f.write('\n\nPer-class Metrics:\n')
            for i in range(num_classes):
                class_name = label_encoder.classes_[i] if i < len(label_encoder.classes_) else f'Class {i}'
                f.write(f'  {class_name:20s}: Precision={metrics["precision_per_class"][i]:.6f}, '
                        f'Recall={metrics["recall_per_class"][i]:.6f}, '
                        f'F1={metrics["f1_per_class"][i]:.6f}, '
                        f'Support={metrics["support_per_class"][i]}\n')
        
        with open(os.path.join(results_dir, 'classification_report.json'), 'w') as f:
            json.dump(metrics['classification_report'], f, indent=4)
        
        class_names = [label_encoder.classes_[i] if i < len(label_encoder.classes_) else f'Class_{i}' 
                      for i in range(num_classes)]
        
        with open(os.path.join(results_dir, 'confusion_matrix.csv'), 'w', newline='') as f:
            writer = csv.writer(f)
            header = ['True/Predicted'] + class_names
            writer.writerow(header)
            
            for i in range(num_classes):
                row = [class_names[i]] + cm[i].tolist()
                writer.writerow(row)
        
        print(f"\n混淆矩阵已保存到: {os.path.join(results_dir, 'confusion_matrix.csv')}")
        
        plt.figure(figsize=(max(12, num_classes), max(10, num_classes)))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                   xticklabels=class_names, yticklabels=class_names,
                   cbar_kws={'label': 'Count'})
        plt.xlabel('Predicted Label', fontsize=12)
        plt.ylabel('True Label', fontsize=12)
        plt.title('Test Confusion Matrix', fontsize=14)
        plt.xticks(rotation=45, ha='right')
        plt.yticks(rotation=0)
        plt.tight_layout()
        plt.savefig(os.path.join(results_dir, 'test_confusion_matrix.png'), dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"混淆矩阵图已保存到: {os.path.join(results_dir, 'test_confusion_matrix.png')}")
    
    return avg_loss, metrics['accuracy']*100, metrics, all_preds, all_labels, all_probs

class StabilityMonitor:
    """稳定性监控器"""
    def __init__(self, window_size=5, threshold=0.2):
        self.window_size = window_size
        self.threshold = threshold
        self.val_losses = []
        self.val_f1_macros = []
    
    def check_stability(self, val_loss, val_f1):
        self.val_losses.append(val_loss)
        self.val_f1_macros.append(val_f1)
        
        if len(self.val_losses) < self.window_size:
            return True, None
        
        recent_losses = self.val_losses[-self.window_size:]
        loss_mean = np.mean(recent_losses)
        loss_std = np.std(recent_losses)
        
        latest_loss = self.val_losses[-1]
        prev_loss = self.val_losses[-2]
        loss_jump = abs(latest_loss - prev_loss) / (prev_loss + 1e-8)
        
        recent_f1s = self.val_f1_macros[-self.window_size:]
        f1_mean = np.mean(recent_f1s)
        f1_std = np.std(recent_f1s)
        
        latest_f1 = self.val_f1_macros[-1]
        
        is_unstable = False
        unstable_reason = None
        
        if loss_jump > self.threshold:
            is_unstable = True
            unstable_reason = f"Loss Jump: {loss_jump:.4f}"
        
        if loss_std > loss_mean * 0.3:
            is_unstable = True
            unstable_reason = f"Loss Oscillation: std/mean = {loss_std/loss_mean:.4f}"
        
        if f1_mean > 0.3 and latest_f1 < f1_mean - f1_std * 2:
            is_unstable = True
            unstable_reason = f"F1 Collapse: {latest_f1:.4f} vs mean {f1_mean:.4f}"
        
        return not is_unstable, unstable_reason

#绘制训练曲线和学习率曲线
def plot_training_curves(results_dir, train_losses, val_losses, train_f1s, val_f1s, 
                         train_accs, val_accs, train_f1_weighted, val_f1_weighted,
                         learning_rates):
    """绘制训练曲线"""
    epochs = range(1, len(train_losses) + 1)
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # 损失曲线
    axes[0, 0].plot(epochs, train_losses, label='Train Loss', linewidth=2, marker='o', markersize=4)
    axes[0, 0].plot(epochs, val_losses, label='Val Loss', linewidth=2, marker='s', markersize=4)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training and Validation Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 准确率曲线
    axes[0, 1].plot(epochs, train_accs, label='Train Accuracy', linewidth=2, marker='o', markersize=4)
    axes[0, 1].plot(epochs, val_accs, label='Val Accuracy', linewidth=2, marker='s', markersize=4)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy')
    axes[0, 1].set_title('Training and Validation Accuracy')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # F1 Macro曲线
    axes[0, 2].plot(epochs, train_f1s, label='Train F1_macro', linewidth=2, marker='o', markersize=4)
    axes[0, 2].plot(epochs, val_f1s, label='Val F1_macro', linewidth=2, marker='s', markersize=4)
    axes[0, 2].set_xlabel('Epoch')
    axes[0, 2].set_ylabel('F1 Score (Macro)')
    axes[0, 2].set_title('Training and Validation F1_macro')
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)
    
    # F1 Weighted曲线
    axes[1, 0].plot(epochs, train_f1_weighted, label='Train F1_weighted', linewidth=2, marker='o', markersize=4)
    axes[1, 0].plot(epochs, val_f1_weighted, label='Val F1_weighted', linewidth=2, marker='s', markersize=4)
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('F1 Score (Weighted)')
    axes[1, 0].set_title('Training and Validation F1_weighted')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # 学习率曲线
    axes[1, 1].plot(epochs[:len(learning_rates)], learning_rates, label='Learning Rate', 
                    linewidth=2, marker='^', markersize=4, color='red')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Learning Rate')
    axes[1, 1].set_title('Learning Rate Schedule')
    axes[1, 1].set_yscale('log')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    # 模型信息
    axes[1, 2].axis('off')
    architecture_text = f"""
     模型: DeepPlantTE
    """
    axes[1, 2].text(0.1, 0.5, architecture_text, fontsize=10, family='monospace',
                    verticalalignment='center', bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
    
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'training_curves.png'), dpi=300)
    plt.close()
    
    # 单独保存学习率曲线
    plt.figure(figsize=(10, 6))
    plt.plot(epochs[:len(learning_rates)], learning_rates, label='Learning Rate', 
             linewidth=3, marker='o', markersize=5, color='darkorange')
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Learning Rate', fontsize=12)
    plt.title('Learning Rate Schedule (Warmup + CosineAnnealing)', fontsize=14)
    plt.yscale('log')
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'learning_rate_schedule.png'), dpi=300)
    plt.close()

#主函数 
def main():
    """主训练函数"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}')
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    set_seed(CONFIG['seed'])
    
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    results_dir = f"results_deepplantte_{timestamp}"
    os.makedirs(results_dir, exist_ok=True)
    
    print("\n" + "="*80)
    print("DeepPlantTE模型")
    
    # 检查数据文件是否存在
    for csv_file in [CONFIG['train_csv'], CONFIG['val_csv'], CONFIG['test_csv']]:
        if not os.path.exists(csv_file):
            print(f"错误: 找不到数据文件 {csv_file}")
            print(f"请确保以下文件存在:")
            print(f"  - {CONFIG['train_csv']}")
            print(f"  - {CONFIG['val_csv']}")
            print(f"  - {CONFIG['test_csv']}")
            return
    
    # 仅从训练集计算序列长度
    train_df = pd.read_csv(CONFIG['train_csv'])
    train_sequences = train_df['Sequence'].apply(len).values
    
    # 使用训练集长度的90%分位数作为统一序列长度
    max_seq_len = int(np.percentile(train_sequences, CONFIG['max_seq_len_percentile']))
    print(f'训练集序列长度统计:')
    print(f'  最小长度: {np.min(train_sequences)}')
    print(f'  最大长度: {np.max(train_sequences)}')
    print(f'  平均长度: {np.mean(train_sequences):.2f}')
    print(f'  中位数长度: {np.median(train_sequences)}')
    print(f'  使用{CONFIG["max_seq_len_percentile"]}%分位数长度: {max_seq_len}')
    print(f'  覆盖的训练序列比例: {(train_sequences <= max_seq_len).mean()*100:.2f}%')
    
    # 统计所有数据集的序列信息
    all_sequences = []
    for csv_file in [CONFIG['train_csv'], CONFIG['val_csv'], CONFIG['test_csv']]:
        df = pd.read_csv(csv_file)
        all_sequences.extend(df['Sequence'].apply(len).values)
    
    sequence_stats = {
        'max_sequence_length': int(np.max(all_sequences)),
        'min_sequence_length': int(np.min(all_sequences)),
        'mean_sequence_length': float(np.mean(all_sequences)),
        'median_sequence_length': float(np.median(all_sequences)),
        'train_percentile_90_length': max_seq_len,
        'percentile_75_length': int(np.percentile(all_sequences, 75)),
        'percentile_95_length': int(np.percentile(all_sequences, 95)),
        'percentile_99_length': int(np.percentile(all_sequences, 99)),
        'total_sequences': len(all_sequences),
        'sequences_per_dataset': {
            'train': len(pd.read_csv(CONFIG['train_csv'])),
            'val': len(pd.read_csv(CONFIG['val_csv'])),
            'test': len(pd.read_csv(CONFIG['test_csv']))
        }
    }
    
    CONFIG['sequence_statistics'] = sequence_stats
    CONFIG['actual_max_seq_len'] = max_seq_len
    
    with open(os.path.join(results_dir, 'config.json'), 'w') as f:
        json.dump(CONFIG, f, indent=4)
    
    # 创建数据集 - 所有数据集使用相同的max_seq_len
    train_dataset = DeepPlantTEDataset(
        CONFIG['train_csv'], max_seq_len, 
        label_encoder=None, augment=True
    )
    train_label_encoder = train_dataset.label_encoder
    
    val_dataset = DeepPlantTEDataset(
        CONFIG['val_csv'], max_seq_len,
        label_encoder=train_label_encoder, augment=False
    )
    test_dataset = DeepPlantTEDataset(
        CONFIG['test_csv'], max_seq_len,
        label_encoder=train_label_encoder, augment=False
    )
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=CONFIG['batch_size'],
        num_workers=2,
        pin_memory=True
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=CONFIG['batch_size'],
        num_workers=2,
        pin_memory=True
    )
    
    num_classes = len(set(train_dataset.labels))
    print(f'\n类别信息:')
    print(f'  类别数量: {num_classes}')
    print(f'  训练集样本数: {len(train_dataset)}')
    print(f'  验证集样本数: {len(val_dataset)}')
    print(f'  测试集样本数: {len(test_dataset)}')
    print(f"  类别: {train_label_encoder.classes_}")
    
    print("\n" + "="*80)
    print("创建DeepPlantTE模型...")
    print("="*80)
    
    model = DeepPlantTE(
        vocab_size=5,
        embed_dim=CONFIG['embed_dim'],
        hidden_dim=CONFIG['hidden_dim'],
        num_classes=num_classes,
        num_lstm_layers=CONFIG['num_lstm_layers'],
        max_seq_len=max_seq_len + 100   # 当前数据集仍按原始 max_seq_len 截断，因此这100个位置并未实际使用。 保留此设置仅为防止未来修改代码时出现越界，属于防御性冗余
    ).to(device)
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'总参数: {total_params:,}')
    print(f'可训练参数: {trainable_params:,}')
    
    # 嵌入层参数统计
    embed_params = sum(p.numel() for p in model.embedding.parameters())
    print(f'嵌入层参数: {embed_params:,} (占总参数的{embed_params/total_params*100:.1f}%)')
    
    criterion = nn.CrossEntropyLoss()
    print(f"使用损失函数: CrossEntropyLoss")
    
    # 使用AdamW优化器
    optimizer = optim.AdamW(
        model.parameters(),
        lr=CONFIG['learning_rate'],
        weight_decay=CONFIG['weight_decay']
    )
    print(f"使用AdamW优化器，权重衰减: {CONFIG['weight_decay']}")
    
    scaler = GradScaler(init_scale=512.0)
    
    print("\n" + "="*80)
    print("学习率调度器配置:")
    print("="*80)
    print(f"1. 预热阶段 (前{CONFIG['warmup_epochs']}个epoch):")
    print(f"   - 线性预热从 {CONFIG['learning_rate']*0.1:.6f} 到 {CONFIG['learning_rate']:.6f}")
    print(f"2. 余弦退火阶段 (从第{CONFIG['warmup_epochs']+1}个epoch开始):")
    print(f"   - T_max = {CONFIG['cosine_T_max']} epochs")
    print(f"   - 最小学习率 = {CONFIG['cosine_eta_min']}")
    print("="*80)
    
    warmup_scheduler = LinearLR(
        optimizer,
        start_factor=0.1,
        total_iters=CONFIG['warmup_epochs']
    )
    
    cosine_scheduler = CosineAnnealingLR(
        optimizer,
        T_max=CONFIG['cosine_T_max'],
        eta_min=CONFIG['cosine_eta_min']
    )
    
    scheduler = SequentialLR(
        optimizer,
        schedulers=[warmup_scheduler, cosine_scheduler],
        milestones=[CONFIG['warmup_epochs']]
    )
    
    stability_monitor = StabilityMonitor(window_size=5, threshold=0.2)
    training_monitor = TrainingMonitor()
    
    best_val_accuracy = 0
    best_val_f1_macro = 0
    best_val_f1_weighted = 0
    best_val_loss = float('inf')
    best_model_path = os.path.join(results_dir, 'best_deepplantte_model.pth')
    early_stop_counter = 0
    
    # 存储所有指标
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    train_f1_macros, val_f1_macros = [], []
    train_f1_weighteds, val_f1_weighteds = [], []
    train_precision_macros, val_precision_macros = [], []
    train_recall_macros, val_recall_macros = [], []
    
    learning_rates = []
    
    print("\n" + "="*80)
    print("开始训练...")
    print(f"序列长度: {max_seq_len}")
    print("="*80 + "\n")
    
    for epoch in range(CONFIG['num_epochs']):
        print(f'\nEpoch {epoch + 1}/{CONFIG["num_epochs"]}')
        print('-'*80)
        
        current_lr = optimizer.param_groups[0]['lr']
        learning_rates.append(current_lr)
        print(f'当前学习率: {current_lr:.6f}')
        
        # 训练
        train_loss, train_metrics = train_epoch(
            model, train_loader, criterion, optimizer, device, scaler,
            epoch, num_classes, train_label_encoder, CONFIG['num_epochs'], results_dir, training_monitor
        )
        
        # 验证
        val_loss, val_metrics = validate(
            model, val_loader, criterion, device, epoch, num_classes,
            train_label_encoder, CONFIG['num_epochs'], results_dir
        )
        
        # 保存所有指标
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        
        train_accs.append(train_metrics['accuracy'])
        val_accs.append(val_metrics['accuracy'])
        
        train_f1_macros.append(train_metrics['f1_macro'])
        val_f1_macros.append(val_metrics['f1_macro'])
        
        train_f1_weighteds.append(train_metrics['f1_weighted'])
        val_f1_weighteds.append(val_metrics['f1_weighted'])
        
        train_precision_macros.append(train_metrics['precision_macro'])
        val_precision_macros.append(val_metrics['precision_macro'])
        
        train_recall_macros.append(train_metrics['recall_macro'])
        val_recall_macros.append(val_metrics['recall_macro'])
        
        scheduler.step()
        training_monitor.track_learning_rate(optimizer)
        
        # 稳定性检查
        if epoch > CONFIG['warmup_epochs'] + 5:
            is_stable, reason = stability_monitor.check_stability(val_loss, val_metrics['f1_macro'])
            if not is_stable:
                print(f'⚠ Warning: {reason}')
                if os.path.exists(best_model_path):
                    model.load_state_dict(torch.load(best_model_path))
                    print(f'已加载最佳模型')
                early_stop_counter += 1
                if early_stop_counter >= 5:
                    print(f'由于训练不稳定，停止训练')
                    break
                continue
        
        # ========== 基于准确率保存最佳模型 ==========
        val_accuracy = val_metrics['accuracy']
        if val_accuracy > best_val_accuracy:  
            best_val_accuracy = val_accuracy
            best_val_f1_macro = val_metrics['f1_macro']  
            best_val_f1_weighted = val_metrics['f1_weighted']
            best_val_loss = val_loss
            torch.save(model.state_dict(), best_model_path)
            
            # 保存最佳模型的详细指标
            best_metrics = {
                'epoch': epoch + 1,
                'train_loss': train_loss,
                'val_loss': val_loss,
                'train_accuracy': train_metrics['accuracy'],
                'val_accuracy': val_metrics['accuracy'],
                'train_f1_macro': train_metrics['f1_macro'],
                'val_f1_macro': val_metrics['f1_macro'],
                'train_f1_weighted': train_metrics['f1_weighted'],
                'val_f1_weighted': val_metrics['f1_weighted'],
                'train_precision_macro': train_metrics['precision_macro'],
                'val_precision_macro': val_metrics['precision_macro'],
                'train_recall_macro': train_metrics['recall_macro'],
                'val_recall_macro': val_metrics['recall_macro'],
            }
            
            with open(os.path.join(results_dir, 'best_model_metrics.json'), 'w') as f:
                json.dump(best_metrics, f, indent=4)
            
            print(f'\n✓ 新的最佳模型已保存! (基于准确率)')
            print(f'  Epoch: {epoch + 1}')
            print(f'  Val Loss: {val_loss:.6f}')
            print(f'  Val Acc: {val_accuracy*100:.4f}%')  
            print(f'  Val F1_macro: {val_metrics["f1_macro"]:.6f}')
            print(f'  Val F1_weighted: {val_metrics["f1_weighted"]:.6f}')
            
            early_stop_counter = 0
        else:
            early_stop_counter += 1
            if early_stop_counter >= CONFIG['early_stop_patience']:
                print(f'\n早停触发于 epoch {epoch + 1}')
                break
    
    torch.cuda.empty_cache()
    
    # 保存所有训练历史指标
    history = {
        'epochs': list(range(1, len(train_losses) + 1)),
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accuracies': train_accs,
        'val_accuracies': val_accs,
        'train_f1_macros': train_f1_macros,
        'val_f1_macros': val_f1_macros,
        'train_f1_weighteds': train_f1_weighteds,
        'val_f1_weighteds': val_f1_weighteds,
        'train_precision_macros': train_precision_macros,
        'val_precision_macros': val_precision_macros,
        'train_recall_macros': train_recall_macros,
        'val_recall_macros': val_recall_macros,
        'learning_rates': learning_rates,
        'best_epoch': train_f1_macros.index(max(train_f1_macros)) + 1 if train_f1_macros else 0,
        'best_val_epoch': val_f1_macros.index(max(val_f1_macros)) + 1 if val_f1_macros else 0,
        'best_val_accuracy_epoch': val_accs.index(max(val_accs)) + 1 if val_accs else 0
    }
    
    with open(os.path.join(results_dir, 'training_history.json'), 'w') as f:
        json.dump(history, f, indent=4)
    
    # 绘制训练曲线
    plot_training_curves(
        results_dir, 
        train_losses, val_losses, 
        train_f1_macros, val_f1_macros,
        train_accs, val_accs,
        train_f1_weighteds, val_f1_weighteds,
        learning_rates
    )
    
    # 保存学习率历史
    with open(os.path.join(results_dir, 'learning_rate_history.json'), 'w') as f:
        json.dump({
            'epochs': list(range(1, len(learning_rates) + 1)),
            'learning_rates': learning_rates,
            'warmup_epochs': CONFIG['warmup_epochs'],
            'cosine_T_max': CONFIG['cosine_T_max'],
            'initial_lr': CONFIG['learning_rate'],
            'scheduler_type': 'Warmup + CosineAnnealingLR',
            'optimizer': f'AdamW (weight_decay={CONFIG["weight_decay"]})'
        }, f, indent=4)
    
    print("\n" + "="*80)
    print("测试最佳模型...")
    print("="*80)
    
    if os.path.exists(best_model_path):
        print(f"加载最佳模型: {best_model_path}")
        model.load_state_dict(torch.load(best_model_path))
        test_loss, test_acc, test_metrics, all_preds, all_labels, all_probs = test(
            model, test_loader, criterion, device, num_classes,
            train_label_encoder, CONFIG['num_epochs'], results_dir
        )
        
        # 保存测试指标
        test_results = {
            'loss': test_loss,
            'accuracy': test_acc / 100,
            'f1_macro': test_metrics['f1_macro'],
            'f1_weighted': test_metrics['f1_weighted'],
            'precision_macro': test_metrics['precision_macro'],
            'recall_macro': test_metrics['recall_macro'],
            'precision_weighted': test_metrics['precision_weighted'],
            'recall_weighted': test_metrics['recall_weighted'],
            'per_class_metrics': {
                'precision': test_metrics['precision_per_class'],
                'recall': test_metrics['recall_per_class'],
                'f1': test_metrics['f1_per_class'],
                'support': test_metrics['support_per_class']
            }
        }
        
        with open(os.path.join(results_dir, 'test_results.json'), 'w') as f:
            json.dump(test_results, f, indent=4)
    else:
        print("未找到最佳模型，使用最终模型进行测试")
        test_loss, test_acc, test_metrics, all_preds, all_labels, all_probs = test(
            model, test_loader, criterion, device, num_classes,
            train_label_encoder, CONFIG['num_epochs'], results_dir
        )
    
    # 保存预测结果
    if results_dir:
        predictions_df = pd.DataFrame({
            'true_label': all_labels,
            'predicted_label': all_preds,
            'true_label_name': [train_label_encoder.inverse_transform([label])[0] for label in all_labels],
            'predicted_label_name': [train_label_encoder.inverse_transform([pred])[0] for pred in all_preds]
        })
        
        for i in range(num_classes):
            class_name = train_label_encoder.classes_[i] if i < len(train_label_encoder.classes_) else f'Class_{i}'
            predictions_df[f'prob_{class_name}'] = [prob[i] for prob in all_probs]
        
        predictions_df.to_csv(os.path.join(results_dir, 'test_predictions.csv'), index=False)
        print(f"预测结果已保存到: {os.path.join(results_dir, 'test_predictions.csv')}")
    
    print("\n" + "="*80)
    print("训练完成!")
    print("="*80)
    print(f"最佳验证准确率: {best_val_accuracy*100:.4f}%") 
    print(f"最佳验证F1_macro: {best_val_f1_macro:.6f}")
    print(f"最佳验证F1_weighted: {best_val_f1_weighted:.6f}")
    print(f"最佳验证损失: {best_val_loss:.6f}")
    print(f"测试准确率: {test_acc:.4f}%")
    print(f"测试F1_macro: {test_metrics['f1_macro']:.6f}")
    print(f"测试F1_weighted: {test_metrics['f1_weighted']:.6f}")
    print(f"\n结果保存在: {results_dir}")
    print("="*80)

if __name__ == "__main__":
    main()

Using device: cuda

DeepPlantTE模型
训练集序列长度统计:
  最小长度: 31
  最大长度: 45892
  平均长度: 1322.62
  中位数长度: 721.0
  使用90%分位数长度: 3107
  覆盖的训练序列比例: 90.01%

类别信息:
  类别数量: 13
  训练集样本数: 214820
  验证集样本数: 26852
  测试集样本数: 26853
  类别: ['DNA/CMC' 'DNA/MULE' 'DNA/PIF' 'DNA/TcMar' 'DNA/hAT' 'LINE/I' 'LINE/L1'
 'LINE/RTE' 'LTR/Caulimovirus' 'LTR/Copia' 'LTR/Gypsy' 'RC/Helitron'
 'SINE/tRNA']

创建DeepPlantTE模型...
总参数: 3,471,150
可训练参数: 3,471,150
嵌入层参数: 482,176 (占总参数的13.9%)
使用损失函数: CrossEntropyLoss


/tmp/ipykernel_96/666161452.py:1216: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(init_scale=512.0)


使用AdamW优化器，权重衰减: 0.02

学习率调度器配置:
1. 预热阶段 (前10个epoch):
   - 线性预热从 0.000100 到 0.001000
2. 余弦退火阶段 (从第11个epoch开始):
   - T_max = 90 epochs
   - 最小学习率 = 1e-05

开始训练...
序列长度: 3107


Epoch 1/100
--------------------------------------------------------------------------------
当前学习率: 0.000100


Training: 100%|██████████| 1679/1679 [10:52<00:00,  2.57it/s, loss=1.6845, acc=46.41%, batch_acc=55.6%, lr=0.000100, skip=G:0/B:0]



训练详细指标 (Epoch 1):
  损失: 1.684451
  准确率: 46.4119%
  ------------------------------
  宏平均指标:
    精确率: 0.247786
    召回率: 0.206382
    F1分数: 0.205519
  ------------------------------
  加权平均指标:
    精确率: 0.427071
    召回率: 0.464119
    F1分数: 0.434211
  ------------------------------
  每个类别指标:
    DNA/CMC             : 精确率=0.291919, 召回率=0.155971, F1=0.203313, 样本数=13573
    DNA/MULE            : 精确率=0.220995, 召回率=0.140553, F1=0.171825, 样本数=16371
    DNA/PIF             : 精确率=0.159587, 召回率=0.016364, F1=0.029685, 样本数=8494
    DNA/TcMar           : 精确率=0.172998, 召回率=0.082936, F1=0.112121, 样本数=3569
    DNA/hAT             : 精确率=0.430966, 召回率=0.553039, F1=0.484431, 样本数=20268
    LINE/I              : 精确率=0.012946, 召回率=0.039347, F1=0.019482, 样本数=1042
    LINE/L1             : 精确率=0.533979, 召回率=0.400484, F1=0.457696, 样本数=15696
    LINE/RTE            : 精确率=0.007757, 召回率=0.006869, F1=0.007286, 样本数=1747
    LTR/Caulimovirus    : 精确率=0.061569, 召回率=0.021939, F1=0.032351, 样本数=2826
    LTR/Copia           

...

Training:  57%|█████▋    | 950/1679 [06:00<04:35,  2.65it/s, loss=0.1243, acc=96.29%, batch_acc=97.7%, lr=0.000034, skip=G:1/B:0] 


⚠ Skipped step due to abnormal grad: nan


Training: 100%|██████████| 1679/1679 [10:36<00:00,  2.64it/s, loss=0.1237, acc=96.30%, batch_acc=100.0%, lr=0.000034, skip=G:1/B:0]



训练详细指标 (Epoch 92):
  损失: 0.123696
  准确率: 96.2978%
  ------------------------------
  宏平均指标:
    精确率: 0.949464
    召回率: 0.941049
    F1分数: 0.945183
  ------------------------------
  加权平均指标:
    精确率: 0.962964
    召回率: 0.962978
    F1分数: 0.962952
  ------------------------------
  每个类别指标:
    DNA/CMC             : 精确率=0.950801, 召回率=0.944006, F1=0.947392, 样本数=13573
    DNA/MULE            : 精确率=0.942774, 召回率=0.945941, F1=0.944355, 样本数=16371
    DNA/PIF             : 精确率=0.955730, 召回率=0.945491, F1=0.950583, 样本数=8494
    DNA/TcMar           : 精确率=0.958939, 召回率=0.942281, F1=0.950537, 样本数=3569
    DNA/hAT             : 精确率=0.969633, 召回率=0.967288, F1=0.968459, 样本数=20268
    LINE/I              : 精确率=0.832075, 召回率=0.846449, F1=0.839201, 样本数=1042
    LINE/L1             : 精确率=0.963958, 召回率=0.957633, F1=0.960785, 样本数=15696
    LINE/RTE            : 精确率=0.961585, 召回率=0.917001, F1=0.938764, 样本数=1747
    LTR/Caulimovirus    : 精确率=0.968501, 召回率=0.946568, F1=0.957409, 样本数=2826
    LTR/Copia          

Training:  13%|█▎        | 218/1679 [01:22<09:23,  2.59it/s, loss=0.1235, acc=96.33%, batch_acc=96.1%, lr=0.000029, skip=G:0/B:0] 